# Mental Health Disorder Detection System

**Model:** DistilBERT (fine-tuned for sequence classification)  
**Dataset:** Reddit Mental Health Dataset  
**Task:** Multi-class text classification — 5 mental health conditions  
**Classes:** Stress · Depression · Bipolar Disorder · Personality Disorder · Anxiety  

---

## Project Overview

Mental health disorders affect hundreds of millions of people worldwide, yet many go undetected due to the stigma around seeking help. Social media platforms like Reddit have become spaces where people openly share their mental health struggles — often using language patterns that are characteristic of specific conditions.

This project fine-tunes **DistilBERT** — a lightweight, efficient transformer model — on Reddit posts to automatically classify text into one of five mental health categories. The goal is to demonstrate that transformer-based NLP can support early detection and awareness of mental health conditions from user-generated text.

### Pipeline
```
Raw Reddit Data  →  Preprocessing  →  DistilBERT Fine-tuning  →  Evaluation  →  Inference
```

---

## 1. Install Dependencies

Install all required libraries. This notebook is designed to run on **Google Colab with GPU (T4 or higher)** enabled.

In [ ]:
!pip install transformers torch accelerate scikit-learn pandas numpy --quiet


## 2. Check GPU Availability

DistilBERT fine-tuning requires a GPU. Verify that Colab has assigned one before proceeding.

> **Tip:** Go to `Runtime → Change runtime type → GPU` if you see `GPU Available: False`.

In [ ]:
import torch

print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')


## 3. Upload Dataset

Upload the **Reddit Mental Health Dataset** CSV file and place it inside a folder called `Reddit Mental Health Data/`.

You can also mount Google Drive if the dataset is already saved there.

In [ ]:
import os

# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Option B: Upload directly
# from google.colab import files
# uploaded = files.upload()

os.makedirs('Reddit Mental Health Data', exist_ok=True)
print('Folder ready. Place your CSV inside: Reddit Mental Health Data/')


## 4. Exploratory Data Analysis (EDA)

Before training, we inspect the dataset to understand its structure, size, and class distribution.
This step helps us identify any data quality issues and confirm the dataset is suitable for training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

file_path = 'Reddit Mental Health Data/Reddit Mental Health Dataset.csv'
df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df.dropna(subset=['title', 'text', 'target'], inplace=True)

print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('First 3 rows:')
df.head(3)


In [ ]:
# Class distribution
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['target'])
class_names = list(label_encoder.classes_)

counts = df['target'].value_counts().reindex(label_encoder.classes_)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_names, counts.values,
              color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Mental Health Condition')
ax.set_ylabel('Number of Posts')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(val), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print('Class counts:')
print(counts.to_string())


In [ ]:
# Text length distribution
df['combined_text'] = df['title'].astype(str) + ' ' + df['text'].astype(str)
df['text_length']   = df['combined_text'].str.split().str.len()

print('Text length statistics (in words):')
print(df['text_length'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['text_length'].clip(upper=600), bins=50, color='steelblue', edgecolor='white')
ax.axvline(df['text_length'].median(), color='red', linestyle='--',
           label=f'Median = {df["text_length"].median():.0f} words')
ax.set_title('Distribution of Post Length (words)', fontsize=14, fontweight='bold')
ax.set_xlabel('Word Count (capped at 600)')
ax.set_ylabel('Number of Posts')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Preprocessing

The preprocessing script performs the following steps:
1. Loads and cleans the raw dataset
2. Combines the `title` and `text` columns into a single input field
3. Encodes string class labels into integers using `LabelEncoder`
4. Saves the label mapping to `label_map.json` so `predict.py` does not need hardcoded labels
5. Splits data into **80% train / 20% test** using stratified sampling
6. Saves all processed files to `processed_data/`

In [ ]:
!python preprocess.py


## 6. Model Training

We fine-tune **DistilBERT** (`distilbert-base-uncased`) for 5-class sequence classification.

### Key training decisions
| Parameter | Value | Reason |
|---|---|---|
| Learning rate | 2e-5 | Standard for BERT fine-tuning |
| Epochs | 4 (early stopping at patience=2) | Prevents overfitting |
| Batch size | 16 | Fits T4 GPU memory |
| Max token length | 512 | Covers most Reddit post lengths |
| Weight decay | 0.01 | L2 regularisation |
| Mixed precision (fp16) | Auto | Faster training on GPU |

Training will take approximately **30–35 minutes** on a Colab T4 GPU.

In [ ]:
!python train.py


## 7. Model Evaluation

After training, we evaluate the best saved model on the held-out test set.

Metrics reported:
- **Accuracy** — overall correct predictions
- **Precision** — how many predicted positives are actually positive
- **Recall** — how many actual positives the model correctly caught
- **F1 Score** — harmonic mean of precision and recall (main metric for multi-class)
- **Confusion Matrix** — per-class breakdown of predictions vs. ground truth

In [ ]:
!python evaluate.py


## 8. Results Summary

The fine-tuned DistilBERT model achieved the following results on the 20% held-out test set:

| Metric | Score |
|---|---|
| Accuracy | **82.89%** |
| Weighted Precision | **83%** |
| Weighted Recall | **83%** |
| Weighted F1 Score | **83%** |

### Per-class performance

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Stress | 0.89 | 0.91 | 0.90 |
| Depression | 0.73 | 0.78 | 0.75 |
| Bipolar Disorder | 0.90 | 0.83 | 0.87 |
| Personality Disorder | 0.80 | 0.81 | 0.80 |
| Anxiety | 0.85 | 0.82 | 0.83 |

**Depression** has the lowest F1 (0.75), which is expected — depressive language overlaps significantly with anxiety and stress. This is a known challenge in mental health NLP.

---

## 9. Inference — Single Text Prediction

Run `predict.py` to classify a custom piece of text. The script outputs:
- The predicted mental health class
- The model's **confidence score** (softmax probability)
- Probabilities for all 5 classes

> **Note:** `predict.py` uses `input()` which requires an interactive terminal. In Colab, it will prompt you to type text after running the cell.

In [ ]:
!python predict.py


## 10. Save Model to Google Drive

Save the trained model to Google Drive so it is not lost when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp -r final_model /content/drive/MyDrive/mental_health_model


Model saved successfully to Google Drive under `MyDrive/mental_health_model/`.

## 11. Limitations & Future Work

### Current Limitations
- **Dataset size:** 5,607 samples is relatively small for transformer fine-tuning. Larger datasets would improve generalisation.
- **Text truncation:** Posts longer than 512 tokens are truncated. Very long posts may lose important context.
- **Domain specificity:** The model is trained only on Reddit data. It may not generalise well to clinical notes or other platforms.
- **Depression misclassification:** Depressive language overlaps with anxiety and stress, causing the lowest per-class F1 (0.75).

### Future Improvements
- Fine-tune on a larger, multi-source dataset (e.g., Twitter, clinical interview transcripts)
- Experiment with full BERT or Mental-BERT — a BERT model pre-trained specifically on mental health corpora
- Add explainability using LIME or SHAP to highlight which words drove the prediction
- Build a web interface using Flask or Streamlit for live demo deployment

---

## References

- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT.* arXiv:1910.01108  
- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers.* arXiv:1810.04805  
- Hugging Face Transformers Library: https://huggingface.co/docs/transformers  
- Reddit Mental Health Dataset: Kaggle  
